# Final Project: Building a Rainfall Prediction Classifier

This notebook builds a machine-learning classifier to predict whether a day will experience heavy rainfall (precipitation > 20 mm) from daily meteorological observations.


## Exercise 1: Import Libraries

Load all required Python libraries.

In [ ]:
import os
import glob
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    roc_auc_score, RocCurveDisplay
)

%matplotlib inline


## Exercise 2: Load the Dataset

For this project we use the `rainfall.csv` dataset which contains daily meteorological observations and a binary target `HeavyRain` (1 if precipitation > 20 mm).

In [ ]:
# Load rainfall.csv from the notebook/course folder.
candidate_paths = [
    "rainfall.csv",
    os.path.join(".", "data", "rainfall.csv"),
]

# Also search subfolders so the notebook can work when the course file is nearby.
candidate_paths.extend(glob.glob("**/rainfall.csv", recursive=True))

data_path = next((p for p in candidate_paths if os.path.isfile(p)), None)

if data_path is None:
    raise FileNotFoundError(
        "rainfall.csv was not found. Place rainfall.csv in the same folder as this notebook "
        "or in a subfolder, then run this cell again."
    )

df = pd.read_csv(data_path)
print("Loaded:", data_path)
df.head()


## Exercise 3: Exploratory Data Analysis (EDA)

Inspect basic statistics, missing values and visualise distributions.

In [ ]:
print("Shape:", df.shape)
display(df.head())
display(df.describe(include="all").T)
print("\nMissing values by column:")
display(df.isna().sum().to_frame("Missing Values"))


In [ ]:
sns.countplot(x="HeavyRain", data=df)
plt.title("Heavy Rain Occurrence")
plt.xlabel("HeavyRain (0 = No Heavy Rain, 1 = Heavy Rain)")
plt.ylabel("Count")
plt.show()


## Exercise 4: Handle Missing Values

We'll fill numeric missing values with the median and categorical with the mode.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

for col in categorical_cols:
    mode = df[col].mode()
    if not mode.empty:
        df[col] = df[col].fillna(mode.iloc[0])

print("Remaining missing values:", int(df.isna().sum().sum()))


## Exercise 5: Feature Engineering

Create additional features such as rolling averages of temperature and humidity.

In [ ]:
df["Temp_3day_avg"] = (
    df["Temperature"].rolling(window=3, min_periods=1).mean()
)
df["Hum_3day_avg"] = (
    df["Humidity"].rolling(window=3, min_periods=1).mean()
)

display(df[["Temperature", "Temp_3day_avg", "Humidity", "Hum_3day_avg"]].head())


## Exercise 6: Define Features and Target

Separate the predictor matrix X and the label y.

In [ ]:
target = "HeavyRain"
features = [col for col in df.columns if col != target]

X = df[features]
y = df[target].astype(int)

print("Features:", features)
print("Target distribution:")
display(y.value_counts().sort_index())


## Exercise 7: Train‑Test Split

Create stratified splits to preserve class balance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))
print("Training class distribution:")
display(y_train.value_counts(normalize=True).sort_index())
print("Testing class distribution:")
display(y_test.value_counts(normalize=True).sort_index())


## Exercise 8: Build a Random Forest Pipeline

Standardise numeric features and train a RandomForest classifier.

In [ ]:
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

numeric_transformer = Pipeline([
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("rf", RandomForestClassifier(random_state=42))
])

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)


## Exercise 9: Baseline Random Forest Training

Fit the pipeline on the training data.

In [ ]:
rf_pipeline.fit(X_train, y_train)

## Exercise 10: Random Forest Evaluation (Test Set)

Calculate accuracy, classification report, confusion matrix and ROC‑AUC.

In [ ]:
y_pred_rf = rf_pipeline.predict(X_test)
y_proba_rf = rf_pipeline.predict_proba(X_test)[:, 1]

rf_acc = accuracy_score(y_test, y_pred_rf)
rf_cm = confusion_matrix(y_test, y_pred_rf)
rf_tpr = (
    rf_cm[1, 1] / (rf_cm[1, 0] + rf_cm[1, 1]) * 100
    if (rf_cm[1, 0] + rf_cm[1, 1]) > 0 else 0
)
rf_roc_auc = roc_auc_score(y_test, y_proba_rf)

print(f"Random Forest Accuracy: {rf_acc:.4f}")
print(f"Random Forest True Positive Rate: {rf_tpr:.0f}%")
print("Random Forest Confusion Matrix:")
print(rf_cm)
print(f"Random Forest ROC-AUC: {rf_roc_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_rf))

RocCurveDisplay.from_predictions(y_test, y_proba_rf)
plt.title("Random Forest ROC Curve")
plt.show()


## Exercise 11: Hyperparameter Tuning (Random Forest)

Search for better `n_estimators` and `max_depth`.

In [ ]:
param_grid = {
    'rf__n_estimators': [100, 200, 300],
    'rf__max_depth': [None, 10, 20, 30]
}
grid = GridSearchCV(rf_pipeline, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)
print('Best RF params:', grid.best_params_)
print('Best CV ROC‑AUC:', grid.best_score_)

## Exercise 12: Retrain Random Forest with Best Params

Create final pipeline and evaluate again.

In [ ]:
# GridSearchCV stores pipeline parameters with the prefix "rf__".
best_rf_params = {
    key.replace("rf__", "", 1): value
    for key, value in grid.best_params_.items()
}

best_rf = Pipeline([
    ("preprocessor", preprocessor),
    ("rf", RandomForestClassifier(**best_rf_params, random_state=42))
])

best_rf.fit(X_train, y_train)

y_pred_best_rf = best_rf.predict(X_test)
y_proba_best_rf = best_rf.predict_proba(X_test)[:, 1]

best_rf_acc = accuracy_score(y_test, y_pred_best_rf)
best_rf_cm = confusion_matrix(y_test, y_pred_best_rf)
best_rf_tpr = (
    best_rf_cm[1, 1] / (best_rf_cm[1, 0] + best_rf_cm[1, 1]) * 100
    if (best_rf_cm[1, 0] + best_rf_cm[1, 1]) > 0 else 0
)
best_rf_roc_auc = roc_auc_score(y_test, y_proba_best_rf)

print("Best RF parameters:", best_rf_params)
print(f"Best RF Accuracy: {best_rf_acc:.4f}")
print(f"Best RF True Positive Rate: {best_rf_tpr:.0f}%")
print("Best RF Confusion Matrix:")
print(best_rf_cm)
print(f"Best RF ROC-AUC: {best_rf_roc_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best_rf))


## Exercise 13: Cross‑validation Score (Best RF)

Report mean ROC‑AUC from 10‑fold CV on the full dataset.

In [ ]:
cv_scores = cross_val_score(
    best_rf,
    X,
    y,
    cv=10,
    scoring="roc_auc",
    n_jobs=-1
)

print("10-fold ROC-AUC scores:", np.round(cv_scores, 4))
print(f"Mean ROC-AUC: {cv_scores.mean():.4f}")


## Exercise 14: Save the Trained Random Forest Model

Persist the final model for later use.

In [ ]:
model_path = "rainfall_classifier.pkl"
joblib.dump(best_rf, model_path)
print("Model saved to:", os.path.abspath(model_path))


---
### Additional Model Comparison: Logistic Regression

This section is included to compare Logistic Regression with the final Random Forest model using the same test split.


In [ ]:
logreg_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("logreg", LogisticRegression(max_iter=1000, random_state=42))
])

logreg_pipeline.fit(X_train, y_train)

y_pred_lr = logreg_pipeline.predict(X_test)
y_proba_lr = logreg_pipeline.predict_proba(X_test)[:, 1]

lr_acc = accuracy_score(y_test, y_pred_lr)
lr_cm = confusion_matrix(y_test, y_pred_lr)
lr_tpr = (
    lr_cm[1, 1] / (lr_cm[1, 0] + lr_cm[1, 1]) * 100
    if (lr_cm[1, 0] + lr_cm[1, 1]) > 0 else 0
)
lr_roc_auc = roc_auc_score(y_test, y_proba_lr)

print(f"Logistic Regression Accuracy: {lr_acc:.4f}")
print(f"Logistic Regression True Positive Rate: {lr_tpr:.0f}%")
print("Logistic Regression Confusion Matrix:")
print(lr_cm)
print(f"Logistic Regression ROC-AUC: {lr_roc_auc:.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_lr))


### Model Comparison Summary

The following table compares the two models on the same test set. Run the notebook to populate the actual metrics.


In [ ]:
summary = pd.DataFrame({
    "Model": ["Random Forest (Best)", "Logistic Regression"],
    "Accuracy": [best_rf_acc, lr_acc],
    "True Positive Rate (%)": [best_rf_tpr, lr_tpr],
    "ROC-AUC": [best_rf_roc_auc, lr_roc_auc]
})

display(summary)


---

### Conclusion

Use the test-set accuracy, true positive rate, and ROC-AUC shown above to describe the two models. Select the model that better matches the assignment's evaluation criterion rather than assuming one model is better before checking the results.
